# 10 · Spark + S3 Object Store — Datalake em RustFS

**Teoria**: docs/08-spark-e-armazenamento-objetos.md

**Pré-requisito**: `make spark && make s3` em execução (ou `make full`).

🎯 **Objetivo**: demonstrar o Spark cluster consumindo o datalake exclusivamente via S3 — sem volume compartilhado com o host, sem depender de disco local dos containers. O cluster é efêmero; a fonte única de verdade é o **RustFS** (S3-compatible object store).

---
### 🔤 O que você vai praticar

1. Upload de dados locais para o bucket `bronze` do RustFS via `boto3`
2. Leitura de Parquet via S3A — `spark.read.parquet("s3a://bronze/vendas")`
3. Select / Filter / WithColumn → **Silver** layer, escrita no S3
4. GroupBy + Agg + OrderBy — agregações de negócio direto no S3
5. Broadcast Join + Window Functions → **Gold** layer, ida e volta pelo S3
6. Spark SQL sobre S3 — `CREATE TEMP VIEW ... LOCATION 's3a://...'`
7. CSV e JSON também no S3 — formatos não-Parquet no object store
8. **Prova da s3ness** — os dados persistem além do container

Vamos começar populando o bucket bronze.

In [ ]:
from pathlib import Path
import boto3
from botocore.config import Config
from IPython.display import clear_output

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="adminpassword",
    config=Config(s3={"addressing_style": "path"}),
    region_name="us-east-1",
)

bronze_dir = Path("../data/bronze")
files = sorted([f for f in bronze_dir.rglob("*") if f.is_file()])
n = len(files)

print(f"📤 Enviando {n} arquivo(s) para s3a://bronze/ ...\n")

for i, fp in enumerate(files, 1):
    key = str(fp.relative_to(bronze_dir))
    s3.upload_file(str(fp), "bronze", key)
    clear_output(wait=True)
    print(f"📤 Upload to s3a://bronze/  [{i:2d}/{n}]")
    print(f"   {key}")

print(f"\n✅ Upload concluído. {n} arquivo(s) em s3a://bronze/")

print("📦 Buckets disponíveis:")
for b in s3.list_buckets()["Buckets"]:
    print(f"🪣  {b['Name']}")

---
### 🧠 O conceito: Spark + S3 sem depender de disco local

Nos notebooks 06-09 o pipeline usava `./data:/data` — os workers enxergavam os arquivos como caminhos locais (`/data/bronze/...`). **Aqui é diferente** (mas os workers ainda têm o volume — ele só não é usado).

Neste perfil **s3**:
- Você se conecta ao **spark-connect-s3** (porta 15003), que carrega as confs S3A apontando pro RustFS
- As confs S3A são propagadas para os workers, que passam a ler/escrever via `s3a://`
- O cluster é **stateless** para o S3 — se morrer, os dados continuam no RustFS
- O volume `./data:/data` continua montado nos workers (não atrapalha — é ignorado)

💡 Essa é a arquitetura real de clouds: Spark no EMR / Dataproc / EKS lêndo e escrevendo em S3 / GCS / ADLS.

Vamos conectar.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("python-app")
    .remote("sc://localhost:15003")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("🖥️  Master UI:    http://localhost:8080")
print("🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082")
print("📊 Spark App UI:  http://localhost:4041")
print("🔌 Spark Connect: sc://localhost:15003  (s3 — sem /data)")

print("⚠️  ATENÇÃO: nenhum volume ./data:/data está montado nos workers.")
print("   Os dados SÓ existem no RustFS S3.")

---
### 📌 Lendo o Bronze diretamente do S3

A leitura usa o mesmo `spark.read.parquet()`, mas o caminho agora é uma URI S3A — o Hadoop S3A connector resolve a comunicação com o RustFS.

💡 O segredo está nas `--conf spark.hadoop.fs.s3a.*` que o `spark-connect-s3` carrega no bootstrap (veja o `docker-compose.yml` profile s3).

In [ ]:
sdf_vendas = spark.read.parquet("s3a://bronze/vendas")
sdf_funcionarios = spark.read.parquet("s3a://bronze/funcionarios")
sdf_empresas = spark.read.parquet("s3a://bronze/empresas")

print("📦 vendas")
sdf_vendas.printSchema()
print(f"   Registros: {sdf_vendas.count():,}")
sdf_vendas.show(5)

print("📦 empresas")
sdf_empresas.show(5)

print("📦 funcionarios")
sdf_funcionarios.show(5)

---
### Select / Filter / WithColumn → Camada Silver

Mesmas transformações dos notebooks 02-04, agora escrevendo no S3.

Vamos enriquecer as vendas com uma faixa de valor (segmentação) e filtrar registros inconsistentes, gravando o resultado particionado por ano no bucket `silver`.

In [ ]:
from pyspark.sql.functions import col, when, sum as spark_sum

vendas_silver = (
    sdf_vendas.filter(col("valor") > 0)
    .withColumn("faixa_valor",
        when(col("valor") < 200, "baixo")
        .when(col("valor") < 1000, "medio")
        .otherwise("alto"))
)

(vendas_silver
    .write.mode("overwrite").partitionBy("ano")
    .parquet("s3a://silver/vendas_enriquecidas"))

sdf_silver = spark.read.parquet("s3a://silver/vendas_enriquecidas")
print(f"✅ Silver escrita: {sdf_silver.count():,} registros em s3a://silver/vendas_enriquecidas")
sdf_silver.select("id_venda", "id_funcionario", "valor", "faixa_valor", "ano", "mes").show(10)

---
### GroupBy + Agg — Análises de Negócio

Agregações clássicas: total de vendas por mês, ticket médio, volume de transações.

📌 Tudo lido e processado diretamente do S3 — o Spark baixa só os arquivos necessários (predicate pushdown via partição `ano`/`mes`).

In [ ]:
from pyspark.sql.functions import avg, count, max as spark_max, round as spark_round

resumo_mensal = (
    sdf_vendas.groupBy("ano", "mes")
    .agg(
        spark_sum("valor").alias("total_vendas"),
        count("*").alias("numero_vendas"),
        spark_round(avg("valor"), 2).alias("ticket_medio"),
        spark_max("valor").alias("maior_venda"),
    )
    .orderBy("ano", "mes")
)

print("📊 Resumo mensal de vendas:")
resumo_mensal.show(15)

# Escreve como camada exploratória no S3
(resumo_mensal
    .write.mode("overwrite").partitionBy("ano")
    .parquet("s3a://silver/resumo_mensal"))

print("✅ Resumo mensal salvo em s3a://silver/resumo_mensal")

---
### Broadcast Join + Window Functions → Camada Gold

Pipeline completo: juntar vendas com empresas (broadcast, já que `empresas` cabe na memória), rankear setores por período, e gravar a **Gold layer** — pronta para dashboards e consultas analíticas.

🧠 Broadcast join evita shuffle: o Spark copia `sdf_empresas` para cada executor via S3 e faz o merge localmente.

In [ ]:
from pyspark.sql.functions import broadcast, row_number
from pyspark.sql.window import Window

vendas_com_setor = (
    sdf_vendas
    .join(broadcast(sdf_empresas), "id_empresa")
)

# Agregação por setor + período
gold_agregado = (
    vendas_com_setor.groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)

print("🏆 Top 10 setores por volume de vendas:")
gold_agregado.show(10)

# Ranking por período (Window)
janela_top = Window.partitionBy("ano", "mes").orderBy(col("total_vendas").desc())
top_setores = (
    gold_agregado
    .withColumn("posicao", row_number().over(janela_top))
    .filter(col("posicao") <= 3)
)

(top_setores
    .write.mode("overwrite").partitionBy("ano", "mes")
    .parquet("s3a://gold/top_setores"))

print("✅ Gold layer escrita em s3a://gold/top_setores")

---
### 🎯 Spark SQL sobre dados do S3

O Spark SQL também funciona perfeitamente com S3. A sintaxe `LOCATION 's3a://...'` cria uma tabela externa apontando direto para o object store.

💡 Isso é o mesmo mecanismo que Hive Metastore / Glue Catalog / Polaris usam para expor datalakes S3 como tabelas SQL.

In [ ]:
# Registra os DataFrames como views temporárias
sdf_vendas.createOrReplaceTempView("vendas")
sdf_empresas.createOrReplaceTempView("empresas")

# Cria uma tabela externa apontando para a Silver no S3
spark.sql("""
    CREATE OR REPLACE TEMP VIEW silver_vendas
    USING parquet
    OPTIONS (path 's3a://silver/vendas_enriquecidas')
""").show()

# Consulta SQL sobre dados no S3
spark.sql("""
    SELECT e.setor, v.ano, v.mes,
           ROUND(SUM(v.valor), 2) AS total
    FROM vendas v
    JOIN empresas e ON v.id_empresa = e.id_empresa
    GROUP BY e.setor, v.ano, v.mes
    ORDER BY total DESC
    LIMIT 10
""").show()

# Explica o plano Catalyst — veja o predicate pushdown!
print("📋 Plano Catalyst da consulta:")
spark.sql("""
    SELECT faixa_valor, ROUND(SUM(valor), 2) AS total
    FROM silver_vendas
    WHERE ano = 2025
    GROUP BY faixa_valor
    ORDER BY total DESC
""").explain(True)

---
### 🎁 Bônus: CSV e JSON também no S3

O S3A connector não serve só para Parquet. Vamos ler as avaliações (`avaliacoes_app`, `avaliacoes_site`, `avaliacoes_callcenter`) diretamente do bucket bronze — exatamente como no notebook 09, mas agora via S3.

📌 Os CSVs com `sep=";"`, `encoding="ISO-8859-1"` e data em formato brasileiro funcionam normalmente — as opções de leitura são as mesmas, independente do storage.

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import LongType, StringType, StructField, StructType

# JSON Lines (app) — campo aninhado
# Schema declarado explicitamente (em vez de inferSchema): evita a passada extra de
# leitura que o Spark faria só para descobrir os tipos (ver notebook 09).
schema_app = StructType([
    StructField("id_avaliacao", LongType()),
    StructField("id_empresa", LongType()),
    StructField("canal", StringType()),
    StructField("nota", LongType()),
    StructField("comentario", StringType()),
    StructField("data", StringType()),
    StructField("dispositivo", StructType([
        StructField("os", StringType()),
        StructField("versao_app", StringType()),
    ])),
])
sdf_app = spark.read.schema(schema_app).json("s3a://bronze/avaliacoes_app")

sdf_app.select("id_avaliacao", "nota", col("dispositivo.os").alias("os")).show(5)

# CSV limpo (site)
sdf_site = (spark.read.option("header", True).option("inferSchema", True)
    .csv("s3a://bronze/avaliacoes_site"))
print("\n📄 Site: {} avaliações".format(sdf_site.count()))
sdf_site.show(5)

# CSV legado (call center) — sep, encoding, data br
sdf_cc = (spark.read.option("header", True)
    .option("sep", ";").option("encoding", "ISO-8859-1").option("inferSchema", True)
    .csv("s3a://bronze/avaliacoes_callcenter"))
print("Call Center: {} avaliações".format(sdf_cc.count()))
sdf_cc.show(5)

# Unifica os 3 canais (unionByName)
sdf_app_norm = sdf_app.select("id_avaliacao", "id_empresa", "nota")
sdf_site_norm = sdf_site.select("id_avaliacao", "id_empresa", "nota")
sdf_cc_norm = sdf_cc.select("id_avaliacao", "id_empresa", "nota")

sdf_todas = (sdf_app_norm
    .unionByName(sdf_site_norm, allowMissingColumns=True)
    .unionByName(sdf_cc_norm, allowMissingColumns=True))

print("Total de avaliações unificadas: {}".format(sdf_todas.count()))


---
### 💪 A prova — dados persistem no S3 além do container

Os dados Gold que escrevemos no S3 **não estão no disco do cluster**. Se derrubarmos os containers Spark e subirmos novos, a gold layer continua lá no RustFS.

Execute mentalmente (ou de verdade):
```bash
make down          # derruba tudo
make spark && make s3   # sobe Spark + RustFS novamente
```

Os dados no RustFS persistiram porque os **volumes dos drives** (`drive0_data` a `drive3_data`) são gerenciados pelo Docker, não pelo Spark.

Vamos verificar lendo a Gold de volta:

In [ ]:
print("📖 Lendo Gold layer do S3...")
sdf_gold = spark.read.parquet("s3a://gold/top_setores")
print(f"   Registros: {sdf_gold.count():,}")
sdf_gold.orderBy("ano", "mes", "posicao").show(15)

# Mostra que o Spark empurra filtros para o S3 (predicate pushdown)
print("📋 Plano Catalyst — veja o pushdown dos filtros:")
sdf_gold.filter(col("ano") == 2025).select("setor", "total_vendas").explain(True)

---
🎉 **Parabéns!** Você completou o notebook 10.

Você aprendeu:
- Popular o bucket `bronze` do RustFS com dados brutos via S3 API (`boto3`)
- Ler e escrever Parquet no S3 com o conector S3A do Hadoop
- Executar **select / filter / withColumn / groupBy / agg / orderBy / join / window** — tudo lendo e escrevendo em S3
- Construir as 3 camadas **Bronze → Silver → Gold** sem depender de disco local dos containers
- Usar **Spark SQL** com `LOCATION 's3a://...'` para consultar dados no object store
- **Provar** que o cluster é stateless — os dados persistem no RustFS além do ciclo de vida do container

🧠 **Moral da história:** num datalake moderno, o Spark é só o motor de processamento. A fonte única de verdade é o **object store** — S3, GCS, ADLS, ou RustFS.

📌 Esse é o mesmo padrão usado por EMR na AWS, Dataproc no GCP e EKS + S3 em arquiteturas cloud-native.

---
**Agora você já domina o pipeline Bronze→Silver→Gold em S3.** No próximo notebook (`11_spark_hdfs_datalake`), você vai executar o mesmo pipeline contra o HDFS (`hdfs://namenode:8020`) e comparar o comportamento dos dois sistemas de armazenamento.